# 🛡️ CrowdShield AI: Interactive Decision-First Walkthrough

**AWS Builder Center Agent Experience Hackathon**  
**Tags:** `#strands-decider-2b` | `#policy`  
**GitHub Repository:** [ALLENKISAIRAKESH/CrowdShield-AI](https://github.com/ALLENKISAIRAKESH/CrowdShield-AI)  

---
## Overview
This notebook demonstrates the **decision-first architecture** of CrowdShield AI:
1. **System One (Policy Classification):** `Strands Decider 2B` performs fast, calibrated choice classification (`NORMAL`, `WARNING`, `CRITICAL`).
2. **Confidence Gate:** Verifies calibrated confidence against an operational threshold (e.g. 70%). Below threshold routes to human review.
3. **System Two (Generative Plan):** An `Amazon Bedrock`-backed Strands Agent synthesizes fact-grounded explanations and operator checklists.
4. **Audit Trace:** Captures every stage from state ingestion to completion.

### 1. Setup & Imports
Import domain schemas, predefined presets, the reusable decision client, and the end-to-end orchestrator.

In [ ]:
import os
import sys
import json
import pandas as pd

# Add project root to Python path
sys.path.append(os.path.abspath('..' if os.path.exists('../src') else '.'))

from src.crowdshield.schemas import (
    CrowdScenario,
    DensityLevel,
    FlowTrend,
    ExitStatus,
    IncidentSignal,
    DecisionStatus,
)
from src.crowdshield.scenarios import PRESET_SCENARIOS
from decision_patterns.crowd_policy.decision import CrowdPolicyDecider
from decision_patterns.crowd_policy.question_definition import get_crowd_policy_question
from src.crowdshield.orchestrator import CrowdShieldOrchestrator

print('✓ CrowdShield AI core modules successfully imported!')

### 2. Inspecting the Reusable Decision Pattern (#policy)
Review the official Strands Decider choice question schema and rubric.

In [ ]:
question = get_crowd_policy_question()
print(json.dumps(question, indent=2))

### 3. Scenario Telemetry & Serialization
Inspect the 4 predefined scenarios and how facts are normalized into a compact state for Strands Decider.

In [ ]:
for key, scenario in PRESET_SCENARIOS.items():
    print(f'=== [{scenario.label}] ===')
    print(scenario.to_state_summary())
    print()

### 4. Running the Policy Classification with Strands Decider 2B
Invoke the reusable `CrowdPolicyDecider` client. If a local Decider server is active at `http://127.0.0.1:8099`, it returns live inference; otherwise, the safety fallback executes gracefully.

In [ ]:
decider = CrowdPolicyDecider(url='http://127.0.0.1:8099', default_threshold=0.70)
sample_state = PRESET_SCENARIOS['growing_congestion'].to_state_summary()

result = decider.evaluate(sample_state)
print('Decider Result:')
print(json.dumps(result, indent=2))

### 5. End-to-End Pipeline Across All Scenarios
Run the full orchestrator: Decider ➔ Confidence Gate ➔ Bedrock Agent ➔ Audit Trace.

In [ ]:
orchestrator = CrowdShieldOrchestrator()
eval_results = []

for key, scenario in PRESET_SCENARIOS.items():
    analysis = orchestrator.analyze(scenario)
    eval_results.append({
        'Scenario': scenario.label,
        'Density': scenario.density_level.value,
        'Exit Status': scenario.exit_status.value,
        'Decision': analysis.decision_result.decision or 'N/A',
        'Status': analysis.decision_result.status.value,
        'Human Review Required': analysis.human_review_required,
        'Total Latency (ms)': analysis.total_elapsed_ms,
    })

df = pd.DataFrame(eval_results)
print(df.to_string(index=False))

### 6. Inspecting the Audit Decision Trace & Action Plan
Examine the generated explanation, checklist, and chronological audit trail for the `blocked_exit` emergency scenario.

In [ ]:
critical_analysis = orchestrator.analyze(PRESET_SCENARIOS['blocked_exit'])

print('=== GENERATED EXPLANATION ===')
print(critical_analysis.explanation)
print('\n=== RECOMMENDED OPERATOR CHECKLIST ===')
for i, step in enumerate(critical_analysis.recommended_steps, 1):
    print(f'{i}. {step}')

print('\n=== AUDIT DECISION TRACE ===')
for event in critical_analysis.trace:
    print(f'[{event.stage.upper()}] {event.message}')

### 7. Exporting Analysis Report (JSON)
The analysis can be exported as a standard JSON report.

In [ ]:
report_json = json.dumps(critical_analysis.model_dump(), indent=2)
with open('sample_crowdshield_report.json', 'w') as f:
    f.write(report_json)

print('✓ Sample report successfully exported to sample_crowdshield_report.json!')